# 🎬 Fallstudie 2: Filmanalyse

**Szenario**: Du analysierst Filmdaten für StreamIt, eine Streaming-Plattform. Fragen: Welches Genre ist am profitabelsten? Gibt es eine Korrelation zwischen Budget und Umsatz?

**Vollständiger DAV-Workflow** – von Rohdaten bis zu Erkenntnissen.

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(BASE_URL + "movies.csv")
print(f'Movies Dataset: {df.shape}')
print(df.head(5))
print()
print(df.describe())

## Phase 2: Data Understanding

In [ ]:
print('Fehlende Werte:')
print(df.isnull().sum())
print()
print('Genre-Verteilung:')
print(df['genre'].value_counts())
print()
print(f'Zeitraum: {df["year"].min()} – {df["year"].max()}')
print(f'Ø Rating: {df["rating"].mean():.2f}')
print(f'Ø Budget: {df["budget_mio"].mean():.1f} Mio $')
print(f'Ø Revenue: {df["revenue_mio"].mean():.1f} Mio $')

In [ ]:
# EDA: Distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0,0].hist(df['rating'], bins=30, color='#0389CF', edgecolor='white', alpha=0.8)
axes[0,0].set_title('Rating-Verteilung', fontweight='bold')
axes[0,0].set_xlabel('Rating (1-10)')

axes[0,1].hist(df['budget_mio'].clip(0,200), bins=30, color='#E8792F', edgecolor='white', alpha=0.8)
axes[0,1].set_title('Budget-Verteilung', fontweight='bold')
axes[0,1].set_xlabel('Mio $')

genre_counts = df['genre'].value_counts()
axes[1,0].bar(genre_counts.index, genre_counts.values, color='#0389CF')
axes[1,0].set_title('Anzahl Filme je Genre', fontweight='bold')
axes[1,0].set_xticklabels(genre_counts.index, rotation=45, ha='right')

year_count = df.groupby('year').size().reset_index(name='count')
axes[1,1].plot(year_count['year'], year_count['count'], color='#0389CF', linewidth=2)
axes[1,1].set_title('Filme pro Jahr', fontweight='bold')
axes[1,1].set_xlabel('Jahr')

plt.tight_layout()
plt.show()

## Phase 3: Data Preparation & Feature Engineering

In [ ]:
df_prep = df.copy()

# Feature Engineering
df_prep['decade'] = (df_prep['year'] // 10) * 10
df_prep['profit_mio'] = df_prep['revenue_mio'] - df_prep['budget_mio']
df_prep['roi'] = (df_prep['profit_mio'] / df_prep['budget_mio'].replace(0,1)).round(3)
df_prep['rating_category'] = pd.cut(df_prep['rating'],
    bins=[0, 5, 6.5, 7.5, 10],
    labels=['Schlecht', 'Mittelmäßig', 'Gut', 'Ausgezeichnet'])
df_prep['blockbuster'] = (df_prep['revenue_mio'] > 100).astype(int)

print('Neue Features:')
print(df_prep[['title','profit_mio','roi','decade','rating_category','blockbuster']].head(8))

## Phase 4: Analyse – Business Questions

In [ ]:
# Frage 1: Welches Genre ist am profitabelsten?
genre_analysis = df_prep.groupby('genre').agg(
    anzahl=('title','count'),
    avg_budget=('budget_mio','mean'),
    avg_revenue=('revenue_mio','mean'),
    avg_profit=('profit_mio','mean'),
    avg_rating=('rating','mean'),
    avg_roi=('roi','mean')
).round(2).sort_values('avg_profit', ascending=False)

print('=== Genre-Analyse ===')
print(genre_analysis.to_string())

In [ ]:
# Interaktive Visualisierung
fig = px.bar(genre_analysis.reset_index(),
    x='genre', y='avg_profit',
    color='avg_rating',
    color_continuous_scale='Blues',
    title='Ø Profit nach Genre (gefärbt nach Ø Rating)',
    labels={'genre':'Genre','avg_profit':'Ø Profit (Mio $)','avg_rating':'Ø Rating'},
    text='avg_profit')
fig.update_traces(texttemplate='$%{text:.0f}M')
fig.update_layout(height=450)
fig.show()

In [ ]:
# Frage 2: Korrelation Budget ↔ Revenue?
print('=== Budget–Revenue Korrelation ===')
corr = df_prep['budget_mio'].corr(df_prep['revenue_mio'])
print(f'Pearson-Korrelation: {corr:.3f}')
print(f'R²: {corr**2:.3f}')
print()
print('Interpretation:', 'starke' if abs(corr)>0.7 else 'moderate' if abs(corr)>0.4 else 'schwache',
      'positive' if corr>0 else 'negative', 'Korrelation')

fig = px.scatter(df_prep,
    x='budget_mio', y='revenue_mio',
    color='genre',
    size='rating',
    hover_data=['title','year'],
    trendline='ols',
    title=f'Budget vs. Revenue (r={corr:.3f})',
    labels={'budget_mio':'Budget (Mio $)','revenue_mio':'Revenue (Mio $)'})
fig.update_layout(height=500)
fig.show()

In [ ]:
# Frage 3: Entwicklung Rating über Dekaden
decade_rating = df_prep.groupby('decade')['rating'].agg(['mean','std','count']).round(2)
print('Rating nach Dekade:')
print(decade_rating)

fig = px.line(decade_rating.reset_index(), x='decade', y='mean',
    title='Ø Filmrating nach Dekade',
    labels={'decade':'Dekade','mean':'Ø Rating'},
    markers=True, line_shape='spline')
fig.update_layout(height=350)
fig.show()

## Insights

1. **Action & Sci-Fi** haben die höchsten absoluten Profite, aber auch die höchsten Budgets
2. **Horror** hat das beste ROI – geringe Produktionskosten, often überraschend hohe Einnahmen
3. **Budget ↔ Revenue** zeigen eine moderate bis starke Korrelation – mehr Budget = tendenziell mehr Einnahmen
4. **Dokumentarfilme** haben niedrige Budgets und Revenues, aber stabile hohe Ratings

---
**Weiter mit:** `11_Fallstudie_Used_Cars.ipynb`